# 03 — Feature Engineering

This notebook transforms the cleaned dataset into model-ready features:
- Define the feature schema (flow-based features from CICIDS2017)
- Select relevant features and drop metadata columns
- Encode categorical labels to integers
- Scale numeric features to zero mean and unit variance
- Visualize engineered feature distributions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from sklearn.preprocessing import LabelEncoder, StandardScaler

sys.path.insert(0, os.path.join(os.path.dirname("__file__" if "__file__" in dir() else ""), ".."))
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

In [ ]:
CLEAN_PATH = os.path.join("..", "data", "processed", "cleaned.parquet")
if os.path.exists(CLEAN_PATH):
    df = pd.read_parquet(CLEAN_PATH)
else:
    # Fallback: load raw and do minimal cleaning inline
    RAW_PATH = os.path.join("..", "data", "raw", "CICIDS2017", "Wednesday-workingHours.pcap_ISCX.csv")
    df = pd.read_csv(RAW_PATH)
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["Label"]).reset_index(drop=True)

print(f"Loaded shape: {df.shape}")

## 1. Feature Schema

In [ ]:
# The 30 flow-based features used by CICIDS2017 for ML experiments
FLOW_FEATURES = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Bwd Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd Header Length",
    "Bwd Header Length",
    "Packet Length Mean",
    "Packet Length Std",
    "ACK Flag Count",
]

available = [f for f in FLOW_FEATURES if f in df.columns]
missing = [f for f in FLOW_FEATURES if f not in df.columns]
print(f"Available features: {len(available)}/{len(FLOW_FEATURES)}")
if missing:
    print(f"Missing from dataset: {missing}")

## 2. Feature Selection

In [ ]:
# Non-feature columns to drop
META_COLS = ["Flow ID", "Source IP", "Source Port", "Destination IP",
             "Destination Port", "Protocol", "Timestamp", "Label"]

feature_cols = [c for c in df.columns if c not in META_COLS and df[c].dtype in ["float64", "int64"]]
print(f"Selected {len(feature_cols)} numeric features")
print("\nFirst 10:", feature_cols[:10])
print("Last 10: ", feature_cols[-10:])

## 3. Label Encoding

In [ ]:
le = LabelEncoder()
df["LabelEncoded"] = le.fit_transform(df["Label"])

print("Label encoding mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls:<25s} -> {i}")

print(f"\nEncoded value counts:")
print(df["LabelEncoded"].value_counts().sort_index().to_string())

## 4. Feature Scaling

In [ ]:
X_raw = df[feature_cols].copy()

# Replace any remaining NaN/inf before scaling
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
X_raw = X_raw.fillna(X_raw.median())

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_raw),
    columns=feature_cols,
    index=df.index,
)

print("Before scaling (sample stats):")
print(X_raw.describe().loc[["mean", "std"]].iloc[:, :5])
print("\nAfter scaling (sample stats):")
print(X_scaled.describe().loc[["mean", "std"]].iloc[:, :5])

## 5. Feature Distributions

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(20, 15))
axes = axes.flatten()
for i, col in enumerate(feature_cols[:30]):
    axes[i].hist(X_scaled[col], bins=30, edgecolor="black", alpha=0.7)
    axes[i].set_title(col, fontsize=8)
for j in range(len(feature_cols[:30]), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Feature Distributions (Scaled)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 6. Engineered Dataset Summary

In [ ]:
print("=" * 50)
print("ENGINEERED DATASET SUMMARY")
print("=" * 50)
print(f"Samples:           {X_scaled.shape[0]:,}")
print(f"Features:          {X_scaled.shape[1]}")
print(f"Label classes:     {len(le.classes_)}")
print(f"NaN in features:   {X_scaled.isnull().sum().sum()}")
print(f"Inf in features:   {X_scaled.isin([np.inf, -np.inf]).sum().sum()}")
print()
print("Label encoding:")
for cls in le.classes_:
    count = (df["Label"] == cls).sum()
    print(f"  {cls:<25s} {count:>10,}")
print("=" * 50)